# 🔴 **Import Libs**

In [ ]:
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 -q
!pip install torchtext==0.18.0 torchmetrics==1.4.0 -q


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torchtext
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

from torch import optim
from torch.nn import functional as F

import tqdm
import torchmetrics as tm

from collections import Counter

In [ ]:
from tqdm import tqdm
for lib in [np, torch, torchtext]:
  print(lib.__name__, '-->', lib.__version__)

In [ ]:
from tqdm import tqdm

# 🔴 **Utils**

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

In [ ]:
def num_trainable_params(model):
  nums = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
  return nums

# 🔴 **Dataset**

## 🟠 **Load the Dataset**

🔰 In this session you should load WikiText2 dataset.

In [ ]:
f1 = open('/kaggle/input/wikitext2/wiki.train.tokens', "rt")
train_text = f1.readlines()
f2 = open('/kaggle/input/wikitext2/wiki.test.tokens', "rt")
test_text = f2.readlines()
f3 = open('/kaggle/input/wikitext2/wiki.valid.tokens', "rt")
valid_text = f3.readlines()


In [ ]:
def convertToList(text):
    res = []
    for i in range(0,len(text)):

        lis = list(text[i].split(' '))
        res.append(lis)
    return res


In [ ]:
train_text[:3]

In [ ]:
train_text = convertToList(train_text)
test_text = convertToList(test_text)
valid_text = convertToList(valid_text)

In [ ]:
train_text[:2]

In [ ]:
def lowercase_texts(texts):
    return [[word.lower() for word in sequence] for sequence in texts]
    
train_text = lowercase_texts(train_text)
test_text = lowercase_texts(test_text)
valid_text = lowercase_texts(valid_text)

train_text[0:3]

## 🟠 **Build vocabulary and save it**

🔰 In this section we need to:

*   Define a tokenizer using `basic_english`
*   Tokenize the dataset and collect tokens
*   Build the vocabulary using `build_vocab_from_iterator`
*   Manually insert special tokens and set the default index


In [ ]:
def jointxt(text):
    return [' '.join(seq) for seq in text];

In [ ]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

tokenizer = get_tokenizer('basic_english')

# Re-join token lists into strings
train_text = jointxt(train_text)
test_text = jointxt(test_text)
valid_text = jointxt(valid_text)

# Build vocab
vocab = build_vocab_from_iterator(map(tokenizer, train_text), specials=['<unk>'])
vocab.set_default_index(vocab['<unk>'])


In [ ]:
vocab(['', 'salam', 'for', 'hello', 'Hi'.lower()])

In [ ]:
torch.save(vocab, '/kaggle/working/vocab.pt')

In [ ]:
len(vocab)

## 🟠 EDA

### 🟡 Let's explore the WikiText2 dataset!

### 🟡 Calculate basic statistics such as the number of documents, total words, average document length, etc.

In [ ]:
total_sentence_count = 0
total_sentence_length = 0

for line in train_text:
    sentences = line.split('.')

    for sentence in sentences:
        tokens = sentence.strip().split()
        sentence_length = len(tokens)

        if sentence_length > 0:
            total_sentence_count += 1
            total_sentence_length += sentence_length

mean_sentence_length = total_sentence_length / total_sentence_count

print(f'Mean sentence length in Wikitext-2: {mean_sentence_length:.2f}')

### 🟡 Analyze the most common and least common words in the dataset.

In [ ]:
freqs = Counter()
for tokens in map(tokenizer, train_text):
  freqs.update(tokens)

In [ ]:
freqs.most_common()[-10000:-9900]

### 🟡  Please proceed with further exploration of the dataset. what do you suggest?

## 🟠 Transform the data

🛑 Make sure to perform the transformations on train, validation and test datasets.

🔰 Reshape the dataset into an `N x B x L` or `M x L` format, where `N` represents the number of batches, `B` is the batch size, `L` is the length of a sample within each batch, and `M` is equal to `N x B`.

In [ ]:
def data_process(raw_text_iter, seq_len):
  data = torch.cat([torch.LongTensor(vocab(tokenizer(line))) for line in raw_text_iter])

  M = len(data) // seq_len

  r = len(data) % seq_len
  data = torch.cat((data, torch.LongTensor([0]))) if r==0 else data

  inputs = data[:M*seq_len]
  inputs = inputs.reshape(-1, seq_len)

  targets = data[1:M*seq_len+1]
  targets = targets.reshape(-1, seq_len)

  return inputs, targets

In [ ]:
inputs, targets = data_process(train_text, seq_len=65)
inputs.shape, targets.shape

## 🟠 Custom dataset

In [ ]:
seq_len = 35

X_train, y_train = data_process(train_text, seq_len)
X_valid, y_valid = data_process(valid_text, seq_len)
X_test, y_test = data_process(test_text, seq_len)

X_train.shape, y_train.shape, X_valid.shape, y_valid.shape, X_test.shape, y_test.shape

🔰 Write a custom dataset class for LanguageModelDataset.

In [ ]:
class LanguageModelDataset(Dataset):

  def __init__(self, inputs, targets):
    self.inputs = inputs
    self.targets = targets

  def __len__(self):
    return self.inputs.shape[0]

  def __getitem__(self, idx):
    return self.inputs[idx], self.targets[idx]

In [ ]:
train_set = LanguageModelDataset(X_train, y_train)
valid_set = LanguageModelDataset(X_valid, y_valid)
test_set = LanguageModelDataset(X_test, y_test)

In [ ]:
train_set[0]

## 🟠 Define a dataloader if needed

🔰 Write dataloaders for the training, validation, and test sets.

In [ ]:
batch_size = 20

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [ ]:
x_batch, y_batch = next(iter(train_loader))
x_batch.shape, y_batch.shape

In [ ]:
train_text[:5]

In [ ]:
from gensim.models import Word2Vec
embedding_dim =  300
# Assuming you trained it already
model = Word2Vec(train_text, vector_size=embedding_dim, window=25, min_count=5, workers=4)
word_vectors = model.wv


In [ ]:
import numpy as np

embedding_dim = 300

stoi = vocab.get_stoi()
embedding_matrix = np.zeros((len(vocab), embedding_dim))

for word, idx in stoi.items():
    if word in word_vectors:
        embedding_matrix[idx] = word_vectors[word]
    else:
        # Word not found in pretrained embeddings — use random vector
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)

# 🔴 **Model**

🔰 Use the following template to create a custom model.

Your model should consist of three parts:

*   an embedding layer
*   an LSTM layer
*   a fully connected layer

In [ ]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers,
                 dropout_embd=0.5, dropout_rnn=0.25, pretrained_embeddings=None,
                 freeze_embeddings=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        if pretrained_embeddings is not None:
            self.embedding.weight = nn.Parameter(torch.tensor(pretrained_embeddings, dtype=torch.float32))
            self.embedding.weight.requires_grad = not freeze_embeddings
        else:
            self.embedding.weight.data.uniform_(-0.1, 0.1)

        self.dropout = nn.Dropout(p=dropout_embd)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            dropout=dropout_rnn, batch_first=True)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src):
        embedding = self.dropout(self.embedding(src))
        output, hidden = self.lstm(embedding)
        prediction = self.fc(output)
        return prediction


In [ ]:
model = LanguageModel(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    hidden_dim=512,
    num_layers=2,
    pretrained_embeddings=embedding_matrix,
    freeze_embeddings=False # or False if you want to fine-tune
).to(device)


In [ ]:
model(x_batch.to(device)).shape

In [ ]:
num_trainable_params(model.embedding), num_trainable_params(model.lstm), num_trainable_params(model.fc)

# 🔴 **Config**

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

🔰 Define the optimizer, loss function, metrics and other necessary parameters in this section, and ensure the model is sent to the appropriate device.

# 🔴 **Train ➰**

🔰 This is the template for train function, change it if needed.

In [ ]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, metric, epoch=None):
  model.train()
  loss_train = AverageMeter()
  metric.reset()

  with tqdm.tqdm(train_loader, unit='batch') as tepoch:
    for inputs, targets in tepoch:
      if epoch:
        tepoch.set_description(f'Epoch {epoch}')

      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)

      loss = loss_fn(outputs.reshape(-1, outputs.shape[-1]), targets.flatten())

      loss.backward()

      nn.utils.clip_grad.clip_grad_norm_(model.parameters(), max_norm=clip)

      optimizer.step()
      optimizer.zero_grad()

      loss_train.update(loss.item(), n=len(targets))
      metric.update(outputs, targets)

      tepoch.set_postfix(loss=loss_train.avg, metric=metric.compute().item())

  return model, loss_train.avg, metric.compute().item()

# 🔴 **Evaluation**

🔰 This is the template for evaluation function, change it if needed.

In [ ]:
def evaluate(model, test_loader, loss_fn, metric):
  model.eval()
  loss_eval = AverageMeter()
  metric.reset()

  with torch.inference_mode():
    for inputs, targets in test_loader:
      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)

      loss = loss_fn(outputs.reshape(-1, outputs.shape[-1]), targets.flatten())
      loss_eval.update(loss.item(), n=len(targets))

      metric(outputs, targets)

  return loss_eval.avg, metric.compute().item()

# 🔴 **Training Process 〽️**

In [ ]:
model = LanguageModel(vocab_size=len(vocab),
                      embedding_dim=300,
                      hidden_dim=512,
                      num_layers=2,
                      dropout_embd=0.25,
                      dropout_rnn=0.25,
                      freeze_embeddings=False,
                       pretrained_embeddings=embedding_matrix).to(device)

In [ ]:
from torch.optim import AdamW , SGD
lr = 1e-5
wd = 1e-4
optimizer = AdamW(model.parameters(), lr= lr, weight_decay=wd)

loss_fn = nn.CrossEntropyLoss()


In [ ]:
loss_train_hist = []
loss_valid_hist = []

metric_train_hist = []
metric_valid_hist = []

best_loss_valid = torch.inf
epoch_counter = 0

In [ ]:
num_epochs = 2
def train(model ,train_loader , val_loader, loss_fn , optimizer):

    global loss_train_hist, loss_valid_hist, metric_train_hist, metric_valid_hist
    global epoch_counter, best_loss_valid 

    for epoch in range(num_epochs):
      # Train
      model, loss_train, metric_train = train_one_epoch(model,
                                                     train_loader,
                                                     loss_fn,
                                                     optimizer,
                                                     metric,
                                                     epoch)
      # Validation
      loss_valid, metric_valid = evaluate(model,
                                         valid_loader,
                                         loss_fn,
                                         metric)
    
      loss_train_hist.append(loss_train)
      loss_valid_hist.append(loss_valid)
    
      metric_train_hist.append(metric_train)
      metric_valid_hist.append(metric_valid)
    
      if loss_valid < best_loss_valid:
        torch.save(model, f'model.pt')
        best_loss_valid = loss_valid
        print('Model Saved!')
    
      print(f'Valid: Loss = {loss_valid:.4}, Metric = {metric_valid:.4}')
      print()
    
      epoch_counter += 1
      

In [ ]:
import tqdm
clip = 0.25

In [ ]:
train(model ,train_loader , valid_loader, loss_fn , optimizer)

## 🟠 Plot

🔰 Plot learning curves

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(range(epoch_counter), loss_train_hist, 'r-', label='Train')
plt.plot(range(epoch_counter), loss_valid_hist, 'b-', label='Validation')

plt.xlabel('Epoch')
plt.ylabel('loss')
plt.grid(True)
plt.legend()

# 🔴 **Test**

🔰 Test your model using data from the test set

# 🔴 **Generate**

🔰 Your mission is to write a `generate` function and use a desired sentence to evaluate the model

In [ ]:
model_path = '/kaggle/working/model.pt'
model = torch.load(model_path)
model.eval()

In [ ]:
def generate(prompt, max_seq_len, temperature, model, tokenizer, vocab, seed=None):
  if seed is not None:
    torch.manual_seed(seed)

  indices = vocab(tokenizer(prompt))
  itos = vocab.get_itos()

  for i in range(max_seq_len):
    src = torch.LongTensor(indices).to(device)

    with torch.no_grad():
      prediction = model(src)

    # Method 1
    # idx = torch.argmax(prediction[-1])
    # itos = vocab.get_itos()
    # itos[idx]

    # Method 2
    probs = torch.softmax(prediction[-1]/temperature, dim=0)

    idx = vocab['<ukn>']
    while idx == vocab['<ukn>']:
      idx = torch.multinomial(probs, num_samples=1).item()

    token = itos[idx]
    prompt += ' ' + token

    if idx == vocab['.']:
      return prompt

    indices.append(idx)

  return prompt

In [ ]:
prompts = ['i want to travel','people play','mike eat banana' , 'what is', 'whay are you' , 'electrical engineers find']

for i in range(len(prompts)):
    print(generate(prompts[i], 35, 0.5, model, tokenizer, vocab))
